# Tập 2: Huấn luyện mô hình đầu tiên của bạn với 100 tấm ảnh
[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ngocnguyen88/AI-from-Zero/blob/main/tap02/tap02.ipynb)

Ở tập 1, mô hình ResNet-18 gọi tấm thẻ mẫu là "phong bì", vì trong 1.000 nhãn nó học không có nhãn nào là thẻ.
Tập này mình **tinh chỉnh** (fine-tuning) chính mô hình đó: giữ nguyên phần "biết nhìn", thay phần "trả lời" ở cuối bằng 3 nhãn mới, rồi dạy bằng 100 tấm ảnh:
`mat-truoc` (mặt trước thẻ mẫu), `mat-sau` (mặt sau thẻ mẫu), `khong-phai-the` (ảnh khác).

Chạy được trên CPU, không cần GPU (Colab miễn phí hoặc máy của bạn), cả notebook vài phút.
Trên Colab: bấm **Runtime > Run all** (Chạy tất cả), hoặc chạy từng ô bằng Shift + Enter.

Thẻ trong bộ ảnh là THẺ MẪU tự vẽ (thông tin bịa, có chữ MẪU), không phải giấy tờ thật. Nguồn và giấy phép từng ảnh: `tap02/DU-LIEU.md`.

**Ô chuẩn bị. Lấy 112 tấm ảnh** (chạy 1 lần, khoảng 4 MB)

In [ ]:
import os, urllib.request, zipfile               # tải bộ ảnh nếu máy chưa có
goc = 'https://raw.githubusercontent.com/ngocnguyen88/AI-from-Zero/main/tap02/'
if not os.path.isdir('du-lieu'):
    for lan in range(3):                         # mạng chập chờn thì thử lại, tối đa 3 lần
        try:
            urllib.request.urlretrieve(goc + 'du-lieu.zip', 'du-lieu.zip.tam')
            os.replace('du-lieu.zip.tam', 'du-lieu.zip')
            break
        except Exception as loi:
            print('Tải lỗi, thử lại:', loi)
    zipfile.ZipFile('du-lieu.zip').extractall('.')
for thu_muc in ['du-lieu/100-anh/mat-truoc', 'du-lieu/100-anh/mat-sau', 'du-lieu/100-anh/khong-phai-the', 'du-lieu/anh-kho']:
    print(len(os.listdir(thu_muc)), 'ảnh trong', thu_muc)

**Ô 1. Nạp 100 tấm ảnh, mỗi thư mục là một nhãn**

In [ ]:
import torch, time
from torch import nn
from torchvision import models, datasets
print('torch', torch.__version__, '| số luồng CPU đang dùng:', torch.get_num_threads())

w = models.ResNet18_Weights.IMAGENET1K_V1                                # bộ số học sẵn, như tập 1
du_lieu = datasets.ImageFolder('du-lieu/100-anh', transform=w.transforms())  # tên thư mục = nhãn
print(len(du_lieu), 'ảnh, các nhãn:', du_lieu.classes)

**Ô 2. Chia 80 ảnh để học, 20 ảnh để kiểm** (ảnh kiểm mô hình không bao giờ được học)

In [ ]:
torch.manual_seed(0)                                       # cố định cách chia, chạy lại vẫn ra như video
hoc, kiem = torch.utils.data.random_split(du_lieu, [80, 20])
nap_hoc = torch.utils.data.DataLoader(hoc, batch_size=16, shuffle=True)
nap_kiem = torch.utils.data.DataLoader(kiem, batch_size=20)
print('ảnh kiểm mỗi nhãn:', {c: sum(du_lieu.targets[i] == k for i in kiem.indices) for k, c in enumerate(du_lieu.classes)})

**Ô 3. Mượn mô hình có sẵn, khoá phần "biết nhìn", thay phần "trả lời"**

In [ ]:
model = models.resnet18(weights=w, progress=False)    # lần đầu tải khoảng 45 MB
for t in model.parameters():
    t.requires_grad = False                    # khoá: 11 triệu số cũ giữ nguyên
model.fc = nn.Linear(512, 3)                   # lớp cuối mới: 512 đặc trưng -> 3 nhãn

so_hoc = sum(t.numel() for t in model.parameters() if t.requires_grad)
so_tong = sum(t.numel() for t in model.parameters())
print(f'{so_hoc} số được học / {so_tong} số trong mô hình ({so_hoc / so_tong * 100:.3f}%)')

**Ô 4. Hàm đo: đoán 20 ảnh kiểm, đếm số ảnh đúng.** Đo thử ngay lúc lớp cuối còn là số ngẫu nhiên.

In [ ]:
def do(model):
    model.eval()                               # chế độ đoán
    dung = 0
    with torch.no_grad():
        for x, y in nap_kiem:
            dung += (model(x).argmax(1) == y).sum().item()
    return dung

print('Trước khi học:', do(model), '/ 20 ảnh kiểm đúng')

**Ô 5. Huấn luyện: 10 vòng, mỗi vòng xem hết 80 ảnh học một lượt**

In [ ]:
toi_uu = torch.optim.Adam(model.fc.parameters(), lr=0.001)   # chỉ chỉnh lớp cuối
ham_sai_so = nn.CrossEntropyLoss()                           # đo mô hình đoán lệch bao nhiêu

t0 = time.time()
for vong in range(1, 11):
    model.train()                              # chế độ học
    tong = 0
    for x, y in nap_hoc:
        loss = ham_sai_so(model(x), y)         # 1. đoán, tính sai số
        toi_uu.zero_grad()
        loss.backward()                        # 2. tính hướng chỉnh từng con số
        toi_uu.step()                          # 3. chỉnh một chút
        tong += loss.item() * len(y)
    print(f'vòng {vong}: sai số {tong / 80:.3f} | ảnh kiểm đúng {do(model)}/20 | {time.time() - t0:.1f} giây')

**Ô 6. Thử ảnh mô hình chưa từng thấy**: tấm thẻ của tập 1, ảnh khó (xoay, tối, nhỏ, bị che), và 3 tấm phong bì

In [ ]:
from PIL import Image

def doan(duong_dan):
    x = w.transforms()(Image.open(duong_dan).convert('RGB')).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        p = model(x).softmax(1)[0]             # 3 điểm -> 3 xác suất
    return du_lieu.classes[p.argmax().item()], p.max().item()

for ten in sorted(os.listdir('du-lieu/anh-kho')):
    nhan, chac = doan('du-lieu/anh-kho/' + ten)
    print(f'{ten:24s} -> {nhan:15s} {chac * 100:5.1f}%')

**Ô 7. Lưu mô hình thành file** (tập 8 sẽ dùng lại file này để làm bản chạy thử)

In [ ]:
torch.save(model.state_dict(), 'mo-hinh-the.pt')
print(round(os.path.getsize('mo-hinh-the.pt') / 1e6, 1), 'MB')

**Bài tự làm (khoảng 10 phút).** Thay 3 thư mục trong `du-lieu/100-anh` bằng 3 loại ảnh của bạn (mỗi loại khoảng 30 tấm, ví dụ: hoá đơn, danh thiếp, ảnh khác), đổi `[80, 20]` ở ô 2 cho khớp tổng số ảnh, rồi chạy lại từ ô 1.
Thử ảnh của riêng bạn: tải ảnh lên (biểu tượng thư mục bên trái trên Colab) rồi chạy `doan('ten-anh-cua-ban.jpg')`.

In [ ]:
# doan('ten-anh-cua-ban.jpg')